# DBO_Quant — NVIDIA GPU Portfolio Optimization

Run this notebook on a remote or on-prem NVIDIA GPU using the **Portfolio Optimization** kernel from NVIDIA's `portfolio-optimization` repository.

Workflow: **Databricks inputs → cuOpt optimization/frontier → backtest → optional rebalancing → push results back to DBO_Quant**.


In [ ]:
from pathlib import Path
import os, sys

DBO_QUANT_ROOT = Path(os.environ.get('DBO_QUANT_ROOT', Path.cwd())).resolve()
for candidate in [DBO_QUANT_ROOT, *DBO_QUANT_ROOT.parents]:
    if (candidate / 'nvidia_bridge').exists() and (candidate / 'gpu').exists():
        DBO_QUANT_ROOT = candidate
        break
sys.path.insert(0, str(DBO_QUANT_ROOT))

from gpu.nvidia_portfolio_optimization.config import load_gpu_config
from gpu.nvidia_portfolio_optimization.runner import run_gpu_workflow
print('DBO_Quant root:', DBO_QUANT_ROOT)


## Configuration
Copy the repository `.env.example` to `.env` and set the `DBO_*` / `DATABRICKS_*` values. The `.env` file is ignored by Git. Environment variables override values in the file.


In [ ]:
CONFIG = load_gpu_config(DBO_QUANT_ROOT)
for key, value in CONFIG.items():
    if key != 'profile':
        print(f'{key}: {value}')


## Run the GPU workflow
This uses NVIDIA's installed `portfolio_optimization` package and requires cuOpt. It will not silently substitute a CPU solver.


In [ ]:
result = run_gpu_workflow(**CONFIG)

display(result['optimal_weights'].sort_values(ascending=False).to_frame())
display(result['frontier'].head(25))
display(result['frontier_figure'])
display(result['backtest_results'])
if result['rebalance_results'] is not None:
    display(result['rebalance_results'])

print('optimization_run_id =', result['optimization_run_id'])
print('rebalance_run_id =', result['rebalance_run_id'])


## Next
Back in Databricks, open `notebooks/portfolio/03_NVIDIA_RESULTS.py` and paste `optimization_run_id`. If rebalancing was enabled, paste `rebalance_run_id` as well. The same results are exposed to OpenBB Workspace by the DBO_Quant App.
